# 04 AI カメラ — IMX500 でオンセンサー物体検出

> **このノートの使い方**：セルを選んで **Shift + Enter** で実行します。上から順に進めてください。
> 緑の **✅ チェックポイント** が出たら次へ。
> 🤖 **困ったら ChatGPT に聞く**（エラーは*全文*を貼る／コードの意味も気軽に質問）。TA への挙手もOK。

## 4-1. なぜ「カメラの中で AI」？
Raspberry Pi AI カメラ（Sony **IMX500**）は、AI 処理回路をセンサに内蔵し、**撮影と推論をセンサの中で同時に**行い、Pi には *結果* だけを返します（エッジ AI）。

- 低遅延 / 低消費電力 / 省帯域 / プライバシー という利点。
- ルールを書く OpenCV と違い、**学習済みモデルが「これは何か」を判断**します。

## 4-2. モデルを載せてカメラを起動
初回は **モデルをセンサへ転送**するため数十秒かかります（進捗バーが出ます）。

In [ ]:
import cv2, time
import matplotlib.pyplot as plt
from IPython.display import clear_output
%matplotlib inline
from picamera2 import Picamera2
from picamera2.devices import IMX500

MODEL = '/usr/share/imx500-models/imx500_network_ssd_mobilenetv2_fpnlite_320x320_pp.rpk'
imx = IMX500(MODEL)
labels = imx.network_intrinsics.labels   # COCO 90 ラベル
picam = Picamera2(imx.camera_num)
picam.configure(picam.create_preview_configuration(
    main={'size':(640,480),'format':'RGB888'}, controls={'FrameRate':15}, buffer_count=8))
imx.show_network_fw_progress_bar(); picam.start(); time.sleep(2)
print('AIカメラ準備OK / ラベル数:', len(labels))

## 4-3. 物体検出（ライブで枠とラベル）
人・椅子・カップ・ペットボトル・本などにカメラを向けてみましょう。

In [ ]:
THRESH = 0.4
N = 40
for i in range(N):
    md_ = picam.capture_metadata()
    outputs = imx.get_outputs(md_, add_batch=True)
    frame = picam.capture_array()
    if outputs is not None:
        boxes, scores, classes = outputs[0][0], outputs[1][0], outputs[2][0]
        for box, score, cls in zip(boxes, scores, classes):
            if score < THRESH: continue
            x,y,w,h = imx.convert_inference_coords(box, md_, picam)
            cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 2)
            cv2.putText(frame, f'{labels[int(cls)]} {score:.2f}', (x,y-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)
    clear_output(wait=True)
    plt.figure(figsize=(7,5)); plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis('off')
    plt.title(f'AI detect {i+1}/{N}'); plt.show()
print('終了')

✅ **チェックポイント**：身近な物に枠とラベル（例 `person 0.82`）が付けば成功です。

- 検出が出ない → 被写体を変える / `THRESH` を 0.3 に下げる。
- `THRESH` を 0.6 に上げると、確信度の高いものだけになります。

> 🤖 **ChatGPT に聞いてみよう**
> 「IMX500 のような**オンセンサーAI（エッジAI）**は、クラウドでAIを動かすのと比べて何が嬉しい？ 具体例つきで」と聞いて、今日のキモを自分の言葉にしてみよう。

## 4-9. 後始末（必ず実行）

In [ ]:
picam.close(); print('カメラ解放')